# TradeFlow AI — nb1_synthetic_generator (Pillar 1: Carrier-Faithful)

**Tujuan**: Men-generate 1.500 dokumen CIPL sintetis yang *strukturnya* identik dengan 5 carrier utama.
**Fix #2**: `tglBl` di Ground Truth sekarang selalu dinormalisasi ke ISO 8601 (YYYY-MM-DD).

In [ ]:
!pip install -q faker reportlab pillow pdf2image

In [ ]:
import os, json, random
from datetime import datetime, timedelta
from dateutil import parser as dateparser
from faker import Faker
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch

fake = Faker('id_ID')
OUTPUT_DIR = './dataset/synthetic'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CARRIERS = ['Hapag-Lloyd', 'MSC', 'Maersk', 'Evergreen', 'Cordelia']
DATE_FORMATS = ['%d/%m/%Y', '%d-%b-%Y', '%B %d, %Y', '%Y-%m-%d', '%d.%m.%Y', '%d %B %Y', '%d-%m-%y']

def parse_to_iso(date_str: str) -> str:
    """Parse any date string to ISO 8601 format YYYY-MM-DD. Fix #2."""
    try:
        return dateparser.parse(date_str, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return '1970-01-01'


In [ ]:
def random_hs_code():
    base = f"{fake.random_int(10, 99)}{fake.random_int(10, 99)}"
    sub  = f"{fake.random_int(10, 99)}"
    ext  = f"{fake.random_int(10, 99)}"
    if random.random() > 0.5:
        return f"{base}.{sub}.{ext}"
    return f"{base}{sub}{ext}"

def random_container():
    prefix = random.choice(['HLXU','MEDU','MSCU','MRSU','EGHU','CRSU'])
    nums   = f"{fake.random_int(1000000, 9999999)}"
    if random.random() < 0.2:
        return f"{prefix} {nums}"
    return f"{prefix}{nums}"

def random_date():
    fmt = random.choice(DATE_FORMATS)
    return fake.date_between(start_date='-5y', end_date='today').strftime(fmt)

def random_weight():
    val = random.uniform(1000.0, 25000.0)
    r   = random.random()
    if r < 0.6:   return f"{val:.2f} KGS", round(val, 2)
    elif r < 0.9: return f"{val:.2f} KGM", round(val, 2)
    else:         return f"{val/1000:.3f} MTS", round(val, 2)


In [ ]:
def draw_carrier_template(c, carrier, data):
    if carrier == 'Hapag-Lloyd':
        c.setFont('Times-Bold', 16)
        c.drawString(0.5*inch, 11*inch, 'Hapag-Lloyd')
        c.setFont('Times-Roman', 10)
        c.drawString(5.5*inch, 11*inch, f"B/L No: {data['nomorBl']}")
        c.drawString(0.5*inch, 9*inch, 'Port of Loading:')
        c.drawString(0.5*inch, 8.8*inch, data['pelabuhan_muat'])
        c.drawString(3.5*inch, 9*inch, 'Port of Discharge:')
        c.drawString(3.5*inch, 8.8*inch, data['pelabuhan_bongkar'])
        c.drawString(0.5*inch, 7*inch, 'Container No.')
        c.drawString(0.5*inch, 6.8*inch, data['container_no'])
        c.drawString(6*inch, 7*inch, 'Gross Weight')
        c.drawString(6*inch, 6.8*inch, data['berat_raw'])
    elif carrier == 'Evergreen':
        c.setFont('Helvetica-Bold', 14)
        c.drawString(3*inch, 11*inch, 'EVERGREEN LINE')
        c.setFont('Helvetica', 8)
        c.drawString(6*inch, 10.5*inch, f"2. B/L No: {data['nomorBl']}")
        c.drawString(0.5*inch, 9*inch, '10. Port of Loading')
        c.drawString(0.5*inch, 8.8*inch, data['pelabuhan_muat'])
        c.drawString(0.5*inch, 8*inch, '11. Port of Discharge')
        c.drawString(0.5*inch, 7.8*inch, data['pelabuhan_bongkar'])
        c.drawString(0.5*inch, 6*inch, '15. Container No.')
        c.drawString(0.5*inch, 5.8*inch, data['container_no'])
        c.drawString(6.5*inch, 6*inch, '19. Gross Weight')
        c.drawString(6.5*inch, 5.8*inch, data['berat_raw'])
    else:
        c.setFont('Helvetica-Bold', 20)
        c.drawString(1*inch, 10.5*inch, f"{carrier} BILL OF LADING")
        c.setFont('Helvetica', 10)
        c.drawString(5.5*inch, 10.5*inch, f"No: {data['nomorBl']}")
        c.drawString(1*inch, 9*inch, f"POL: {data['pelabuhan_muat']}")
        c.drawString(4*inch, 9*inch, f"POD: {data['pelabuhan_bongkar']}")
        c.drawString(1*inch, 7*inch, f"CNTR: {data['container_no']}")
        c.drawString(4*inch, 7*inch, f"WT: {data['berat_raw']}")
    c.setFont('Helvetica', 9)
    c.drawString(0.5*inch, 5*inch, 'DESCRIPTION OF GOODS:')
    c.drawString(0.5*inch, 4.8*inch, f"{fake.sentence()} HS: {data['hs_code']}")
    c.drawString(5*inch, 2*inch, f"Date: {data['tglBl_raw']}")


In [ ]:
def generate_synthetic_doc(idx):
    carrier    = random.choice(CARRIERS)
    bl_number  = f"{carrier[:3].upper()}{fake.random_int(1000000, 9999999)}"
    date_raw   = random_date()
    date_iso   = parse_to_iso(date_raw)   # FIX #2: selalu simpan ISO 8601
    weight_raw, weight_norm = random_weight()
    cntr       = random_container()
    hs         = random_hs_code()

    data = {
        'nomorBl':       bl_number,
        'tglBl_raw':     date_raw,
        'pelabuhan_muat': fake.city().upper() + ' PORT',
        'pelabuhan_bongkar': fake.city().upper() + ' PORT',
        'container_no':  cntr,
        'berat_raw':     weight_raw,
        'beratKotor':    weight_norm,
        'hs_code':       hs
    }

    pdf_path = os.path.join(OUTPUT_DIR, f'synth_{idx:04d}.pdf')
    c = canvas.Canvas(pdf_path, pagesize=A4)
    draw_carrier_template(c, carrier, data)
    c.save()

    gt = {
        'document_type': 'bill_of_lading',
        'carrier':       carrier,
        'nomorBl':       data['nomorBl'],
        'tglBl':         date_iso,          # FIX #2: YYYY-MM-DD, bukan date_raw
        'pelabuhan_muat': data['pelabuhan_muat'],
        'pelabuhan_bongkar': data['pelabuhan_bongkar'],
        'container_no':  data['container_no'].replace(' ', ''),
        'beratKotor':    data['beratKotor'],
        'hs_code':       data['hs_code'].replace('.', '')
    }
    with open(os.path.join(OUTPUT_DIR, f'synth_{idx:04d}.json'), 'w') as f:
        json.dump(gt, f, indent=2)

print('Generating 1,500 carrier-faithful synthetic documents...')
for i in range(1500):
    generate_synthetic_doc(i)
print('Done! Output:', OUTPUT_DIR)
